# 06 — Kotekar Ablation Training (KAGGLE GPU)
## WavSent-MTL · Tasks 3.1–3.12

**Runs on: Kaggle T4 x2 GPU**

**What this notebook does:**
- Configs A, B, C → 30 seeds each (TKAN encoder)
- After A/B/C: compare B vs C mean val accuracy → set BEST_REPR
- Configs D (LSTM), E (GRU), F (TCN) → 30 seeds each using BEST_REPR input
- Saves best-seed val+test predictions per model (for PSO in notebook 08)
- Saves results CSV after every config (crash-safe)

**PREREQUISITE:** config/config.py `best_params` must be updated from notebook 05 output

**Input shapes:**
- Config A (returns + polarity_mean): n_features varies (computed inside notebook)
- Config B (denoised OHLCV + polarity_mean): 6 features
- Config C (denoised technicals + polarity_mean): 8 features (selected_features)
- Configs D/E/F: BEST_REPR features

In [ ]:
# ── Kaggle Setup ────────────────────────────────────────────────────────────
import subprocess
subprocess.run(['git', 'clone', 'https://github.com/IAMNEERAJ05/WavSent-MTL.git',
                '/kaggle/working/WavSent-MTL'], check=True)

import sys
import os
sys.path.insert(0, '/kaggle/working/WavSent-MTL')
os.chdir('/kaggle/working/WavSent-MTL')

import torch
import gc
from config.config import CONFIG

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch: {torch.__version__}')
print(f'Device:  {device}')
if device == 'cuda':
    print(f'GPU:     {torch.cuda.get_device_name(0)}')

print()
print('─' * 45)
print('Training Configuration')
print('─' * 45)
print(f"Early stopping patience : {CONFIG['early_stopping_patience']}")
print(f"Early stopping monitor  : {CONFIG['early_stopping_monitor']}")
print(f"Max epochs              : {CONFIG['max_epochs']}")
print(f"LR reduce patience      : {CONFIG['lr_reduce_patience']}")
print(f"LR reduce monitor       : {CONFIG['lr_reduce_monitor']}")
print(f"LR reduce factor        : {CONFIG['lr_reduce_factor']}")
print(f"Loss type               : {CONFIG['loss_type']}")
print('─' * 45)

## Load Kotekar Processed Arrays

In [ ]:
import numpy as np
import pandas as pd
import json

KOTEKAR_INPUT = '/kaggle/input/wavsent-kotekar-processed'

# Full data dict (Config C / selected features — 8 features)
data_C = {
    'X_train':      np.load(f'{KOTEKAR_INPUT}/X_train.npy'),
    'X_val':        np.load(f'{KOTEKAR_INPUT}/X_val.npy'),
    'X_test':       np.load(f'{KOTEKAR_INPUT}/X_test.npy'),
    'y_clf_train':  np.load(f'{KOTEKAR_INPUT}/y_clf_train.npy'),
    'y_clf_val':    np.load(f'{KOTEKAR_INPUT}/y_clf_val.npy'),
    'y_clf_test':   np.load(f'{KOTEKAR_INPUT}/y_clf_test.npy'),
    'y_reg_train':  np.load(f'{KOTEKAR_INPUT}/y_reg_train.npy'),
    'y_reg_val':    np.load(f'{KOTEKAR_INPUT}/y_reg_val.npy'),
    'y_reg_test':   np.load(f'{KOTEKAR_INPUT}/y_reg_test.npy'),
}

with open(f'{KOTEKAR_INPUT}/selected_features.json') as f:
    selected_features = json.load(f)   # 8 features

with open(f'{KOTEKAR_INPUT}/class_weights.json') as f:
    class_weights = json.load(f)       # None or {0: w0, 1: w1}

print(f'X_train shape:     {data_C["X_train"].shape}')
print(f'Selected features: {selected_features}')
print(f'Class weights:     {class_weights}')

## Build Config A and B Data Variants

Config A uses daily returns + polarity_mean (Kotekar-spirit baseline).  
Config B uses denoised OHLCV (5 cols) + polarity_mean = 6 features.  
Both are subsets of featured_data.csv — reconstruct from the full arrays.

In [ ]:
# We need featured_data.csv to rebuild Config A and B inputs.
# Upload data/processed/kotekar/featured_data.csv as part of the Kaggle dataset.
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from src.data.windows import create_windows, generate_targets, temporal_split
from src.data.preprocessor import apply_scaler, apply_reg_scaler

df_feat = pd.read_csv(f'{KOTEKAR_INPUT}/featured_data.csv')
df_feat['Date'] = pd.to_datetime(df_feat['Date'])
df_feat = df_feat.sort_values('Date').reset_index(drop=True)

# Temporal split (same split as notebook 04)
train_df, val_df, test_df = temporal_split(df_feat)
print(f'Split: Train={len(train_df)} Val={len(val_df)} Test={len(test_df)}')

def build_data_variant(feature_cols, train_df, val_df, test_df, w=5):
    """Scale, window, and target-generate for given feature columns."""
    tr_s, v_s, te_s, _ = apply_scaler(
        train_df[feature_cols].values,
        val_df[feature_cols].values,
        test_df[feature_cols].values,
        save_path='/tmp/tmp_scaler.pkl'
    )
    X_train = create_windows(tr_s, w)
    X_val   = create_windows(v_s, w)
    X_test  = create_windows(te_s, w)
    y_clf_tr, y_reg_tr = generate_targets(train_df['Close'].values, w)
    y_clf_v,  y_reg_v  = generate_targets(val_df['Close'].values, w)
    y_clf_te, y_reg_te = generate_targets(test_df['Close'].values, w)
    y_reg_tr, y_reg_v, y_reg_te, _ = apply_reg_scaler(
        y_reg_tr, y_reg_v, y_reg_te, save_path='/tmp/tmp_reg_scaler.pkl')
    return {
        'X_train': X_train, 'X_val': X_val, 'X_test': X_test,
        'y_clf_train': y_clf_tr, 'y_clf_val': y_clf_v, 'y_clf_test': y_clf_te,
        'y_reg_train': y_reg_tr, 'y_reg_val': y_reg_v, 'y_reg_test': y_reg_te,
    }

# Config A: daily Close return + polarity_mean
# Compute return series: (Close_t - Close_t-1) / Close_t-1
for split_df in [train_df, val_df, test_df]:
    split_df['return'] = split_df['Close'].pct_change().fillna(0)

CONFIG_A_FEATURES = ['return', 'polarity_mean']
data_A = build_data_variant(CONFIG_A_FEATURES, train_df, val_df, test_df)
print(f'Config A X_train: {data_A["X_train"].shape}  (2 features)')

# Config B: denoised OHLCV (5) + polarity_mean = 6 features
CONFIG_B_FEATURES = ['Open_d', 'High_d', 'Low_d', 'Close_d', 'Volume_d', 'polarity_mean']
data_B = build_data_variant(CONFIG_B_FEATURES, train_df, val_df, test_df)
print(f'Config B X_train: {data_B["X_train"].shape}  (6 features)')

print(f'Config C X_train: {data_C["X_train"].shape}  (8 features — already loaded)')

## Helper: Run One Config

In [ ]:
from src.training.trainer import train_multi_run

RESULTS_PATH = '/kaggle/working/kotekar_ablation_partial.csv'
all_results = pd.DataFrame()


def run_config(config_name, model_name, data, class_weights=None):
    """Run 30-seed training for one config and append to results CSV."""
    global all_results
    n_features = data['X_train'].shape[2]
    print(f'\n{"=" * 60}')
    print(f'Config {config_name} | {model_name.upper()} | n_features={n_features}')
    print(f'{"=" * 60}')

    results = train_multi_run(
        config_name=config_name,
        model_name=model_name,
        n_features=n_features,
        data=data,
        dataset='kotekar',
        class_weights=class_weights,
        device=device,
    )

    all_results = pd.concat([all_results, results], ignore_index=True)
    all_results.to_csv(RESULTS_PATH, index=False)
    print(f'Results saved ({len(all_results)} rows total)')

    mean_val = results['val_accuracy'].mean()
    mean_test = results['accuracy'].mean()
    print(f'Config {config_name} | mean val_acc={mean_val:.4f} | mean test_acc={mean_test:.4f}')

    torch.cuda.empty_cache()
    gc.collect()
    return results

## Tasks 3.2–3.4 — Run Configs A, B, C (TKAN, 30 seeds each)

In [ ]:
results_A = run_config('A', 'tkan', data_A, class_weights)

In [ ]:
results_B = run_config('B', 'tkan', data_B, class_weights)

In [ ]:
results_C = run_config('C', 'tkan', data_C, class_weights)

## Task 3.5 — Compare B vs C → Set BEST_REPR

> **ACTION REQUIRED:** After running this cell, manually update `BEST_REPR` in `config/config.py` and push to GitHub before running D/E/F.

In [ ]:
mean_val_B = results_B['val_accuracy'].mean()
mean_val_C = results_C['val_accuracy'].mean()

print('=' * 50)
print('B vs C Comparison (mean val accuracy, 30 seeds)')
print('=' * 50)
print(f'Config B (denoised OHLCV):         {mean_val_B:.4f}')
print(f'Config C (denoised technicals):    {mean_val_C:.4f}')

winner = 'denoised_ohlcv' if mean_val_B > mean_val_C else 'denoised_technicals'
winner_data = data_B if mean_val_B > mean_val_C else data_C
winner_label = 'B' if mean_val_B > mean_val_C else 'C'

print(f'\nWINNER: Config {winner_label} -> BEST_REPR = "{winner}"')
print()
print('ACTION: Update config/config.py:')
print(f'    \'BEST_REPR\': \'{winner}\',')
print('Then push to GitHub and re-clone before running D/E/F.')

## Task 3.6 — Reload Updated Config After BEST_REPR is Set

> Re-clone from GitHub after updating config.py and rerun this cell before D/E/F.

In [ ]:
# After updating BEST_REPR in config.py and pushing to GitHub:
# subprocess.run(['git', '-C', '/kaggle/working/WavSent-MTL', 'pull'], check=True)

import importlib
import config.config as _cfg_mod
importlib.reload(_cfg_mod)
from config.config import CONFIG

BEST_REPR = CONFIG['BEST_REPR']
print(f'BEST_REPR = "{BEST_REPR}"')

# Select data for BEST_REPR
if BEST_REPR == 'denoised_ohlcv':
    data_BEST = data_B
elif BEST_REPR == 'denoised_technicals':
    data_BEST = data_C
else:
    raise ValueError(f'Unknown BEST_REPR: {BEST_REPR}')

print(f'Using data_BEST: X_train shape = {data_BEST["X_train"].shape}')

## Tasks 3.7–3.9 — Run Configs D (LSTM), E (GRU), F (TCN)

In [ ]:
results_D = run_config('D', 'lstm', data_BEST, class_weights)

In [ ]:
results_E = run_config('E', 'gru', data_BEST, class_weights)

In [ ]:
results_F = run_config('F', 'tcn', data_BEST, class_weights)

## Summary

In [ ]:
# Save final complete results
final_path = '/kaggle/working/kotekar_ablation_AG.csv'
all_results.to_csv(final_path, index=False)

print('=' * 60)
print('Notebook 06 -- Kotekar Training: COMPLETE')
print('=' * 60)

summary = all_results.groupby('config').agg(
    mean_val_acc=('val_accuracy', 'mean'),
    mean_test_acc=('accuracy', 'mean'),
    std_test_acc=('accuracy', 'std'),
    mean_auc=('auc', 'mean'),
).round(4)
print(summary.to_string())

print(f'\nDownload these files from /kaggle/working/:')
print('  kotekar_ablation_AG.csv')
print('  kotekar_ablation_partial.csv  (incremental backup)')
print()
print('  kotekar_tkan_best.pt          ← best TKAN weights (for SHAP)')
print('  kotekar_lstm_best.pt          ← best LSTM weights (for SHAP)')
print('  kotekar_gru_best.pt           ← best GRU  weights (for SHAP)')
print('  kotekar_tcn_best.pt           ← best TCN  weights (for SHAP)')
print()
print('Val predictions saved at:')
print('  ablation/results/kotekar/val_predictions/*.npy')
print()
print('After downloading .pt files, place them at:')
print('  results/saved_models/kotekar/{model}_best.pt')
print()
print('Next: run notebook 07_training_kaggle')